# Ativos com pelo menos 10 anos de dados de preços e ITR e ordene pelo volume financeiro médio.

In [1]:
from pipelines.readers.pipelines.cvm_formulario_informacoes_trimestrais.reader_parquet import ReaderSnapshotParquet as itr
from data_providers.providers.yfinance_price_provider import YFinancePriceProvider as YFProvider
from pipelines.readers.pipelines.b3_indices_segmentos_setoriais.reader_parquet import ReaderSnapshotParquet as b3_indices
from pipelines.readers.pipelines.b3_enriquecimento_cadastral_ativos.reader_parquet import ReaderSnapshotParquet as b3_enriquecimento
from pandas import DataFrame, to_datetime

In [2]:
b3_index_df = b3_indices(file_identifiers="composicao.parquet").read().copy()

In [3]:
b3_index_select = (
    b3_index_df[b3_index_df["index"] == "IBOV"]
    .reset_index(drop=True)
)

In [ ]:
b3_index_select.tail(3)

,segment,cod,asset,type,part,partAcum,theoricalQty,index
0,<NA>,ALOS3,ALLOS,ON NM,0.531,NaN,478975645,IBOV
1,<NA>,ABEV3,AMBEV S/A,ON,2.450,NaN,4043349180,IBOV
2,<NA>,ASAI3,ASSAI,ON NM,0.467,NaN,1335651252,IBOV
3,<NA>,AURE3,AUREN,ON NM,0.147,NaN,317886569,IBOV
4,<NA>,AXIA3,AXIA ENERGIA,ON NM,4.849,NaN,2278697967,IBOV
...,...,...,...,...,...,...,...,...
71,<NA>,VAMO3,VAMOS,ON NM,0.071,NaN,527021942,IBOV
72,<NA>,VBBR3,VIBRA,ON NM,1.700,NaN,1194079963,IBOV
73,<NA>,VIVA3,VIVARA S.A.,ON NM,0.110,NaN,123861677,IBOV
74,<NA>,WEGE3,WEG,ON NM,2.909,NaN,1460506056,IBOV


In [5]:
b3_index_select.shape

(76, 8)

In [6]:
price_stats = {}

for row in b3_index_select.itertuples():
    
    price_df = YFProvider().get_asset_price(tickers=row.cod+".SA", period="max", interval="1d").copy()
    
    price_stats[row.cod] = {
        "date_start_price": price_df.index.min(),
        "date_end_price": price_df.index.max(),
        "ma_volume_financeiro": price_df["Volume"].mean()
        }


In [7]:
price_stats_df = (
    DataFrame.from_dict(price_stats, orient="index")
    .rename_axis("cod")
    .reset_index()
)

assets_with_price_stats_df = b3_index_select.merge(price_stats_df, on="cod", how="left")

In [8]:
assets_with_price_stats_df.tail(3)

,segment,cod,asset,type,part,partAcum,theoricalQty,index,date_start_price,date_end_price,ma_volume_financeiro
73,<NA>,VIVA3,VIVARA S.A.,ON NM,0.110,NaN,123861677,IBOV,2019-10-11,2026-09-04,2.294756e+06
74,<NA>,WEGE3,WEG,ON NM,2.909,NaN,1460506056,IBOV,2000-01-03,2026-09-04,4.008667e+06
75,<NA>,YDUQ3,YDUQS PART,ON NM,0.093,NaN,247852006,IBOV,2008-07-11,2026-09-04,2.865740e+06


In [9]:
cvm_codes_df = b3_enriquecimento(file_identifiers="codigos.parquet").read()
cvm_codes_df.rename(columns={"code": "cod"}, inplace=True)

In [10]:
cvm_codes_df.tail(3)

,cod,isin,codeCVM
519,FRAS3,BRFRASACNOR0,6211
520,DOHL3,BRDOHLACNOR2,5207
521,DOHL4,BRDOHLACNPR9,5207


In [11]:
assets_with_cvm_codes_df = assets_with_price_stats_df.merge(cvm_codes_df, on="cod", how="left")
assets_with_cvm_codes_df.tail(3)

,segment,cod,asset,type,part,partAcum,theoricalQty,index,date_start_price,date_end_price,ma_volume_financeiro,isin,codeCVM
73,<NA>,VIVA3,VIVARA S.A.,ON NM,0.110,NaN,123861677,IBOV,2019-10-11,2026-09-04,2.294756e+06,BRVIVAACNOR0,24805
74,<NA>,WEGE3,WEG,ON NM,2.909,NaN,1460506056,IBOV,2000-01-03,2026-09-04,4.008667e+06,BRWEGEACNOR0,5410
75,<NA>,YDUQ3,YDUQS PART,ON NM,0.093,NaN,247852006,IBOV,2008-07-11,2026-09-04,2.865740e+06,BRYDUQACNOR3,21016


In [12]:
itr_stats = {}

for row in assets_with_cvm_codes_df.itertuples():
    
    itr_df = itr("BPP_con").query_parquet(filters={"CD_CVM": str(row.codeCVM).zfill(6)})
    
    date_start = itr_df["DT_REFER"].min()
    date_end = itr_df["DT_REFER"].max()
    
    itr_stats[row.codeCVM] = {"date_start_itr": date_start, "date_end_itr": date_end}

In [13]:
itr_stats_df = (
    DataFrame.from_dict(itr_stats, orient="index")
    .rename_axis("codeCVM")
    .reset_index()
)

In [14]:
itr_stats_df.tail(3)

,codeCVM,date_start_itr,date_end_itr
71,24805,2019-06-30,2026-06-30
72,5410,2011-03-31,2026-06-30
73,21016,2011-03-31,2026-06-30


In [15]:
eligible_assets_df = assets_with_cvm_codes_df.merge(itr_stats_df, on="codeCVM", how="left")
eligible_assets_df.tail(3)

,segment,cod,asset,type,part,partAcum,theoricalQty,index,date_start_price,date_end_price,ma_volume_financeiro,isin,codeCVM,date_start_itr,date_end_itr
73,<NA>,VIVA3,VIVARA S.A.,ON NM,0.110,NaN,123861677,IBOV,2019-10-11,2026-09-04,2.294756e+06,BRVIVAACNOR0,24805,2019-06-30,2026-06-30
74,<NA>,WEGE3,WEG,ON NM,2.909,NaN,1460506056,IBOV,2000-01-03,2026-09-04,4.008667e+06,BRWEGEACNOR0,5410,2011-03-31,2026-06-30
75,<NA>,YDUQ3,YDUQS PART,ON NM,0.093,NaN,247852006,IBOV,2008-07-11,2026-09-04,2.865740e+06,BRYDUQACNOR3,21016,2011-03-31,2026-06-30


## Critério de elegibilidade

Filtra os ativos do índice que possuem:
- **Histórico de preços ≥ 10 anos** (Yahoo Finance)
- **Histórico de ITR ≥ 10 anos** (dados trimestrais CVM)

Resultado ordenado por volume financeiro médio (desc).

In [16]:
diff_dias_preco = to_datetime(eligible_assets_df["date_end_price"], format="%Y-%m-%d") - eligible_assets_df["date_start_price"]
eligible_assets_df["anos_diferenca_preco"] = diff_dias_preco.dt.days / 365.25

In [17]:
diff_dias_itr = to_datetime(eligible_assets_df["date_end_itr"], format="%Y-%m-%d") - eligible_assets_df["date_start_itr"]
eligible_assets_df["anos_diferenca_itr"] = diff_dias_itr.dt.days / 365.25

In [18]:
eligible_assets_df.tail(3)

,segment,cod,asset,type,part,partAcum,theoricalQty,index,date_start_price,date_end_price,ma_volume_financeiro,isin,codeCVM,date_start_itr,date_end_itr,anos_diferenca_preco,anos_diferenca_itr
73,<NA>,VIVA3,VIVARA S.A.,ON NM,0.110,NaN,123861677,IBOV,2019-10-11,2026-09-04,2.294756e+06,BRVIVAACNOR0,24805,2019-06-30,2026-06-30,6.899384,7.000684
74,<NA>,WEGE3,WEG,ON NM,2.909,NaN,1460506056,IBOV,2000-01-03,2026-09-04,4.008667e+06,BRWEGEACNOR0,5410,2011-03-31,2026-06-30,26.669405,15.249829
75,<NA>,YDUQ3,YDUQS PART,ON NM,0.093,NaN,247852006,IBOV,2008-07-11,2026-09-04,2.865740e+06,BRYDUQACNOR3,21016,2011-03-31,2026-06-30,18.149213,15.249829


In [19]:
eligible_assets_df[
    (eligible_assets_df["anos_diferenca_itr"] >= 10) & (eligible_assets_df["anos_diferenca_preco"] >= 10)
].sort_values(by="ma_volume_financeiro", ascending=False)[["cod", "asset", "codeCVM", "anos_diferenca_preco", "anos_diferenca_itr", "ma_volume_financeiro"]].reset_index(drop=True)

,cod,asset,codeCVM,anos_diferenca_preco,anos_diferenca_itr,ma_volume_financeiro
0,PETR4,PETROBRAS,9512,26.669405,15.249829,4.886559e+08
1,BRAP4,BRADESPAR,18724,26.680356,12.501027,2.567677e+08
2,PETR3,PETROBRAS,9512,26.669405,15.249829,7.105836e+07
3,ITUB4,ITAUUNIBANCO,19348,25.702943,15.249829,4.712252e+07
4,B3SA3,B3,21610,18.869268,15.249829,3.335611e+07
5,COGN3,COGNA ON,17973,14.475017,15.249829,2.524775e+07
6,ITSA4,ITAUSA,7617,26.680356,15.249829,2.213156e+07
7,CSAN3,COSAN,19836,20.804928,15.000684,1.481531e+07
8,USIM5,USIMINAS,14320,26.680356,15.249829,1.449792e+07
9,LREN3,LOJAS RENNER,8133,26.677618,15.249829,1.348869e+07
